# Instacart Data Check

This notebook performs data checks before segmentation and basket creation.

## Goals
- Validate key relationships between tables
- Check duplicates and value ranges
- Verify consistency between orders and products tables
- Confirm that prior/train orders cover departments and aisles
- Prepare a clean data quality report

## Output
This notebook produces simple validation tables used before segmentation and rule mining.

In [3]:
import pandas as pd
import numpy as np

In [4]:
def load_instacart_tables():
    """
    Load the main Instacart tables.
    """
    tables = {
        "orders": pd.read_csv("../data/orders.csv"),
        "order_products__prior": pd.read_csv("../data/order_products__prior.csv"),
        "order_products__train": pd.read_csv("../data/order_products__train.csv"),
        "products": pd.read_csv("../data/products.csv"),
        "aisles": pd.read_csv("../data/aisles.csv"),
        "departments": pd.read_csv("../data/departments.csv"),
    }
    return tables


def enrich_products(products, aisles, departments):
    """
    Merge products with aisle and department names.
    """
    df = products.merge(aisles, on="aisle_id", how="left")
    df = df.merge(departments, on="department_id", how="left")
    return df


def basic_table_check(df, name):
    """
    Build a simple table quality summary.
    """
    row = {
        "table": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    }
    return pd.DataFrame([row])


def key_uniqueness_check(orders, products, aisles, departments):
    """
    Check uniqueness of primary keys.
    """
    rows = []

    rows.append({
        "table": "orders",
        "key_column": "order_id",
        "n_rows": len(orders),
        "n_unique_keys": orders["order_id"].nunique(),
        "duplicate_key_rows": int(orders["order_id"].duplicated().sum()),
    })

    rows.append({
        "table": "products",
        "key_column": "product_id",
        "n_rows": len(products),
        "n_unique_keys": products["product_id"].nunique(),
        "duplicate_key_rows": int(products["product_id"].duplicated().sum()),
    })

    rows.append({
        "table": "aisles",
        "key_column": "aisle_id",
        "n_rows": len(aisles),
        "n_unique_keys": aisles["aisle_id"].nunique(),
        "duplicate_key_rows": int(aisles["aisle_id"].duplicated().sum()),
    })

    rows.append({
        "table": "departments",
        "key_column": "department_id",
        "n_rows": len(departments),
        "n_unique_keys": departments["department_id"].nunique(),
        "duplicate_key_rows": int(departments["department_id"].duplicated().sum()),
    })

    return pd.DataFrame(rows)


def foreign_key_check(orders, op_prior, op_train, products):
    """
    Check order_id and product_id consistency in order_products tables.
    """
    order_ids = set(orders["order_id"].unique())
    product_ids = set(products["product_id"].unique())

    rows = []

    for name, df in [
        ("order_products__prior", op_prior),
        ("order_products__train", op_train),
    ]:
        missing_orders = int((~df["order_id"].isin(order_ids)).sum())
        missing_products = int((~df["product_id"].isin(product_ids)).sum())

        rows.append({
            "table": name,
            "missing_order_id_in_orders": missing_orders,
            "missing_product_id_in_products": missing_products,
        })

    return pd.DataFrame(rows)


def check_orders_value_ranges(orders):
    """
    Check valid ranges in orders table.
    """
    ds = orders["days_since_prior_order"]

    row = {
        "bad_order_dow": int((~orders["order_dow"].between(0, 6)).sum()),
        "bad_order_hour_of_day": int((~orders["order_hour_of_day"].between(0, 23)).sum()),
        "days_since_prior_order_missing": int(ds.isna().sum()),
        "bad_days_since_prior_order_non_missing": int((~ds.dropna().between(0, 30)).sum()),
        "bad_eval_set_values": int((~orders["eval_set"].isin(["prior", "train", "test"])).sum()),
    }

    return pd.DataFrame([row])


def duplicate_line_check(op_prior, op_train):
    """
    Check duplicate (order_id, product_id) pairs.
    """
    prior_dup_pairs = int(op_prior.duplicated(subset=["order_id", "product_id"]).sum())
    train_dup_pairs = int(op_train.duplicated(subset=["order_id", "product_id"]).sum())

    out = pd.DataFrame([
        {
            "table": "order_products__prior",
            "duplicate_order_product_pairs": prior_dup_pairs,
        },
        {
            "table": "order_products__train",
            "duplicate_order_product_pairs": train_dup_pairs,
        },
    ])
    return out


def add_eval_set_to_order_products(orders, op_df):
    """
    Attach eval_set to an order_products table.
    """
    out = op_df.merge(
        orders[["order_id", "eval_set", "user_id", "order_number"]],
        on="order_id",
        how="left",
    )
    return out


def eval_set_alignment_check(orders, op_prior, op_train):
    """
    Check that prior lines match prior orders and train lines match train orders.
    """
    prior_join = add_eval_set_to_order_products(orders, op_prior)
    train_join = add_eval_set_to_order_products(orders, op_train)

    prior_bad = int((prior_join["eval_set"] != "prior").sum())
    train_bad = int((train_join["eval_set"] != "train").sum())

    return pd.DataFrame([
        {"table": "order_products__prior", "rows_with_wrong_eval_set": prior_bad},
        {"table": "order_products__train", "rows_with_wrong_eval_set": train_bad},
    ])


def department_aisle_coverage(op_df, products_enriched, label):
    """
    Check department and aisle coverage for an order_products table.
    """
    tmp = op_df.merge(
        products_enriched[["product_id", "department", "aisle"]],
        on="product_id",
        how="left",
    )

    n_lines = len(tmp)
    n_products = tmp["product_id"].nunique()
    n_departments = tmp["department"].nunique(dropna=True)
    n_aisles = tmp["aisle"].nunique(dropna=True)

    missing_department = int(tmp["department"].isna().sum())
    missing_aisle = int(tmp["aisle"].isna().sum())

    out = pd.DataFrame([{
        "table": label,
        "n_lines": n_lines,
        "n_products": n_products,
        "n_departments_seen": n_departments,
        "n_aisles_seen": n_aisles,
        "missing_department_rows": missing_department,
        "missing_aisle_rows": missing_aisle,
    }])

    return out


def compare_catalog_coverage(op_prior, op_train, products_enriched):
    """
    Compare catalog coverage between prior and train.
    """
    prior_tmp = op_prior.merge(
        products_enriched[["product_id", "department", "aisle"]],
        on="product_id",
        how="left",
    )
    train_tmp = op_train.merge(
        products_enriched[["product_id", "department", "aisle"]],
        on="product_id",
        how="left",
    )

    prior_deps = set(prior_tmp["department"].dropna().unique())
    train_deps = set(train_tmp["department"].dropna().unique())

    prior_aisles = set(prior_tmp["aisle"].dropna().unique())
    train_aisles = set(train_tmp["aisle"].dropna().unique())

    out = pd.DataFrame([{
        "departments_in_prior": len(prior_deps),
        "departments_in_train": len(train_deps),
        "departments_only_in_prior": len(prior_deps - train_deps),
        "departments_only_in_train": len(train_deps - prior_deps),
        "aisles_in_prior": len(prior_aisles),
        "aisles_in_train": len(train_aisles),
        "aisles_only_in_prior": len(prior_aisles - train_aisles),
        "aisles_only_in_train": len(train_aisles - prior_aisles),
    }])

    return out


def user_order_sequence_check(orders):
    """
    Check chronological consistency per user using order_number.
    """
    tmp = orders.sort_values(["user_id", "order_number"]).copy()

    # Count users with repeated order_number
    repeated_pairs = tmp.duplicated(subset=["user_id", "order_number"]).sum()

    # Compare max order_number with count of orders per user
    user_stats = tmp.groupby("user_id").agg(
        n_orders=("order_id", "count"),
        max_order_number=("order_number", "max"),
        min_order_number=("order_number", "min"),
    ).reset_index()

    user_stats["starts_at_1"] = user_stats["min_order_number"] == 1
    user_stats["count_matches_max"] = user_stats["n_orders"] == user_stats["max_order_number"]

    out = pd.DataFrame([{
        "n_users": user_stats["user_id"].nunique(),
        "users_not_starting_at_1": int((~user_stats["starts_at_1"]).sum()),
        "users_order_count_mismatch_max_order_number": int((~user_stats["count_matches_max"]).sum()),
        "duplicate_user_order_number_pairs": int(repeated_pairs),
    }])

    return out


def reorder_flag_check(op_prior, op_train):
    """
    Check values of reordered flag.
    """
    rows = []
    for name, df in [("order_products__prior", op_prior), ("order_products__train", op_train)]:
        bad_values = int((~df["reordered"].isin([0, 1])).sum())
        rows.append({
            "table": name,
            "bad_reordered_values": bad_values,
            "reordered_rate": float(df["reordered"].mean()),
        })
    return pd.DataFrame(rows)


def add_to_cart_order_check(op_prior, op_train):
    """
    Check add_to_cart_order values.
    """
    rows = []
    for name, df in [("order_products__prior", op_prior), ("order_products__train", op_train)]:
        bad_non_positive = int((df["add_to_cart_order"] <= 0).sum())
        missing_vals = int(df["add_to_cart_order"].isna().sum())
        rows.append({
            "table": name,
            "bad_add_to_cart_order_non_positive": bad_non_positive,
            "missing_add_to_cart_order": missing_vals,
            "max_add_to_cart_order": int(df["add_to_cart_order"].max()),
        })
    return pd.DataFrame(rows)

In [ ]:
# Load data
tables = load_instacart_tables()

orders = tables["orders"]
op_prior = tables["order_products__prior"]
op_train = tables["order_products__train"]
products = tables["products"]
aisles = tables["aisles"]
departments = tables["departments"]

products_enriched = enrich_products(products, aisles, departments)

In [6]:
# Basic quality checks
basic_checks = pd.concat(
    [
        basic_table_check(orders, "orders"),
        basic_table_check(op_prior, "order_products__prior"),
        basic_table_check(op_train, "order_products__train"),
        basic_table_check(products, "products"),
        basic_table_check(aisles, "aisles"),
        basic_table_check(departments, "departments"),
    ],
    ignore_index=True,
)
display(basic_checks)

display(key_uniqueness_check(orders, products, aisles, departments))
display(foreign_key_check(orders, op_prior, op_train, products))

,table,rows,cols,missing_cells,duplicate_rows
0,orders,3421083,7,206209,0
1,order_products__prior,32434489,4,0,0
2,order_products__train,1384617,4,0,0
3,products,49688,4,0,0
4,aisles,134,2,0,0
5,departments,21,2,0,0


,table,key_column,n_rows,n_unique_keys,duplicate_key_rows
0,orders,order_id,3421083,3421083,0
1,products,product_id,49688,49688,0
2,aisles,aisle_id,134,134,0
3,departments,department_id,21,21,0


,table,missing_order_id_in_orders,missing_product_id_in_products
0,order_products__prior,0,0
1,order_products__train,0,0


In [7]:
# Value and format checks
display(check_orders_value_ranges(orders))
display(duplicate_line_check(op_prior, op_train))
display(eval_set_alignment_check(orders, op_prior, op_train))
display(reorder_flag_check(op_prior, op_train))
display(add_to_cart_order_check(op_prior, op_train))

,bad_order_dow,bad_order_hour_of_day,days_since_prior_order_missing,bad_days_since_prior_order_non_missing,bad_eval_set_values
0,0,0,206209,0,0


,table,duplicate_order_product_pairs
0,order_products__prior,0
1,order_products__train,0


,table,rows_with_wrong_eval_set
0,order_products__prior,0
1,order_products__train,0


,table,bad_reordered_values,reordered_rate
0,order_products__prior,0,0.589697
1,order_products__train,0,0.598594


,table,bad_add_to_cart_order_non_positive,missing_add_to_cart_order,max_add_to_cart_order
0,order_products__prior,0,0,145
1,order_products__train,0,0,80


In [8]:
# Coverage checks (departments and aisles)
coverage_prior = department_aisle_coverage(
    op_prior,
    products_enriched,
    "order_products__prior",
)
coverage_train = department_aisle_coverage(
    op_train,
    products_enriched,
    "order_products__train",
)

display(pd.concat([coverage_prior, coverage_train], ignore_index=True))
display(compare_catalog_coverage(op_prior, op_train, products_enriched))

,table,n_lines,n_products,n_departments_seen,n_aisles_seen,missing_department_rows,missing_aisle_rows
0,order_products__prior,32434489,49677,21,134,0,0
1,order_products__train,1384617,39123,21,134,0,0


,departments_in_prior,departments_in_train,departments_only_in_prior,departments_only_in_train,aisles_in_prior,aisles_in_train,aisles_only_in_prior,aisles_only_in_train
0,21,21,0,0,134,134,0,0


In [9]:
# Detailed department coverage
prior_dep = op_prior.merge(
    products_enriched[["product_id", "department"]],
    on="product_id",
    how="left",
).groupby("department").size().reset_index(name="n_lines_prior")

train_dep = op_train.merge(
    products_enriched[["product_id", "department"]],
    on="product_id",
    how="left",
).groupby("department").size().reset_index(name="n_lines_train")

dep_compare = prior_dep.merge(train_dep, on="department", how="outer").fillna(0)
dep_compare["n_lines_prior"] = dep_compare["n_lines_prior"].astype(int)
dep_compare["n_lines_train"] = dep_compare["n_lines_train"].astype(int)
dep_compare = dep_compare.sort_values("n_lines_prior", ascending=False).reset_index(drop=True)
display(dep_compare)

,department,n_lines_prior,n_lines_train
0,produce,9479291,409087
1,dairy eggs,5414016,217051
2,snacks,2887550,118862
3,beverages,2690129,114046
4,frozen,2236432,100426
5,pantry,1875577,81242
6,bakery,1176787,48394
7,canned goods,1068058,46799
8,deli,1051249,44291
9,dry goods pasta,866627,38713


In [10]:
# Detailed aisle coverage (top aisles)
prior_aisle = op_prior.merge(
    products_enriched[["product_id", "aisle"]],
    on="product_id",
    how="left",
).groupby("aisle").size().reset_index(name="n_lines_prior")

train_aisle = op_train.merge(
    products_enriched[["product_id", "aisle"]],
    on="product_id",
    how="left",
).groupby("aisle").size().reset_index(name="n_lines_train")

aisle_compare = prior_aisle.merge(train_aisle, on="aisle", how="outer").fillna(0)
aisle_compare["n_lines_prior"] = aisle_compare["n_lines_prior"].astype(int)
aisle_compare["n_lines_train"] = aisle_compare["n_lines_train"].astype(int)
aisle_compare = aisle_compare.sort_values("n_lines_prior", ascending=False).reset_index(drop=True)
display(aisle_compare.head(30))

,aisle,n_lines_prior,n_lines_train
0,fresh fruits,3642188,150473
1,fresh vegetables,3418021,150609
2,packaged vegetables fruits,1765313,78493
3,yogurt,1452343,55240
4,packaged cheese,979763,41699
5,milk,891015,32644
6,water seltzer sparkling water,841533,36617
7,chips pretzels,722470,31269
8,soy lactosefree,638253,26240
9,bread,584834,23635


In [11]:
# User order sequence checks
display(user_order_sequence_check(orders))

,n_users,users_not_starting_at_1,users_order_count_mismatch_max_order_number,duplicate_user_order_number_pairs
0,206209,0,0,0


In [12]:
# Distribution of eval_set by user (simple check)
user_eval = orders.pivot_table(
    index="user_id",
    columns="eval_set",
    values="order_id",
    aggfunc="count",
    fill_value=0,
).reset_index()

for col in ["prior", "train", "test"]:
    if col not in user_eval.columns:
        user_eval[col] = 0

user_eval_summary = pd.DataFrame([{
    "n_users_total": int(user_eval["user_id"].nunique()),
    "users_with_prior_orders": int((user_eval["prior"] > 0).sum()),
    "users_with_train_order": int((user_eval["train"] > 0).sum()),
    "users_with_test_order": int((user_eval["test"] > 0).sum()),
    "users_with_both_train_and_test": int(((user_eval["train"] > 0) & (user_eval["test"] > 0)).sum()),
}])

display(user_eval_summary)

,n_users_total,users_with_prior_orders,users_with_train_order,users_with_test_order,users_with_both_train_and_test
0,206209,206209,131209,75000,0


In [13]:
# Final data check summary
final_summary = pd.DataFrame([{
    "foreign_keys_ok": bool(
        (foreign_key_check(orders, op_prior, op_train, products)[
            ["missing_order_id_in_orders", "missing_product_id_in_products"]
        ].sum().sum()) == 0
    ),
    "eval_alignment_ok": bool(
        (eval_set_alignment_check(orders, op_prior, op_train)["rows_with_wrong_eval_set"].sum()) == 0
    ),
    "order_ranges_ok": bool(
        (check_orders_value_ranges(orders)[[
            "bad_order_dow",
            "bad_order_hour_of_day",
            "bad_days_since_prior_order_non_missing",
            "bad_eval_set_values",
        ]].sum().sum()) == 0
    ),
    "duplicate_order_product_pairs_prior": int(
        duplicate_line_check(op_prior, op_train).iloc[0]["duplicate_order_product_pairs"]
    ),
    "duplicate_order_product_pairs_train": int(
        duplicate_line_check(op_prior, op_train).iloc[1]["duplicate_order_product_pairs"]
    ),
}])

display(final_summary)

,foreign_keys_ok,eval_alignment_ok,order_ranges_ok,duplicate_order_product_pairs_prior,duplicate_order_product_pairs_train
0,True,True,True,0,0
